# COFOG-Panel: Harmonizing IMF GFS COFOG Data into Reproducible Panel Datasets

This notebook automates the setup, installation, and execution of the COFOG-Panel pipeline based on the project [README](README.md).

In [ ]:
# @title 1. Setup Environment and Install Dependencies
# @markdown This cell installs dependencies and sets up the package in editable mode.

!git clone https://github.com/royalgarter/cofog-panel.git
%cd cofog-panel

# Install dependencies from requirements.txt
!pip install -r requirements.txt

# Install the package in editable mode to use the `cofog-panel` CLI command
!pip install -e .

## 2.1. Prepare Data

Ensure your input files are placed in the `./data/` directory:
- `gfs_raw_data.xlsx` (Main COFOG source)
- `country_codes.xlsx` (Country mapping file)

In [3]:
import os

# Create necessary directories
os.makedirs('data', exist_ok=True)
os.makedirs('output', exist_ok=True)
os.makedirs('intermediate_splits', exist_ok=True)

print("Directories verified.")

Directories verified.


# 2.2. Data Form

* **'COFOG' Dropdown**: This dropdown lists all the unique COFOG (Classification of the Functions of Government) values found in your gfs_raw_data.xlsx file. You can select a specific spending function from this list.

* **'TYPE_OF_TRANSFORMATION' Dropdown**: This dropdown shows the unique data transformation types (e.g., 'Percent of total outlays', 'Percent of GDP') available in your data. You can choose how the data is represented.

* **'SECTOR' Dropdown**: This dropdown contains all the unique sectors (e.g., 'Central government', 'Local Government') present in your Excel file. You can filter the data by a specific sector.

* **'Data Column Name' Text Input**: This is an empty box where you can type a custom name for a new data column that will be created later in the processing pipeline.

In [18]:
import ipywidgets as widgets
from IPython.display import display
import pandas as pd

# Define the path to the raw GFS data Excel file
file_path = './data/gfs_raw_data.xlsx'
# Load the data into a pandas DataFrame
df = pd.read_excel(file_path)

# Create Dropdown for COFOG values
cofog_values = df['COFOG'].unique()
dropdown_cofog = widgets.Dropdown(
    options=cofog_values,
    description='Choose COFOG:',
)
display(dropdown_cofog)

# Create Dropdown for TYPE_OF_TRANSFORMATION values
transformation_values = df['TYPE_OF_TRANSFORMATION'].unique()
dropdown_transformartion = widgets.Dropdown(
    options=transformation_values,
    description='Choose Transformation Type:',
)
display(dropdown_transformartion)

# Create Dropdown for SECTOR values
sector_values = df['SECTOR'].unique()
dropdown_sector = widgets.Dropdown(
    options=sector_values,
    description='Choose Sector:',
)
display(dropdown_sector)

# Create Text Input for new data column name
input_data_col = widgets.Text(
    placeholder='Enter new data column name here',
    description='Data Column Name:'
)
display(input_data_col)

Dropdown(description='Choose COFOG:', options=('Fuel and energy', 'Tertiary education', 'Social protection', '…

Dropdown(description='Choose Transformation Type:', options=('Percent of total outlays', 'Percent of GDP', 'Do…

Dropdown(description='Choose Sector:', options=('Social security funds', 'State Government', 'Extrabudgetary c…

Text(value='', description='Data Column Name:', placeholder='Enter new data column name here')

In [19]:
print(input_data_col.value)

fdfd


## 3. Run Pipeline Stages

You can run the stages individually or use the orchestrator.

In [ ]:
# @title Orchestrator: Run Full Pipeline
# @markdown This command automates Stages 1 through 5 sequentially.

!cofog-panel run \
    --source-file ./data/gfs_raw_data.xlsx \
    --lookup-file ./data/country_codes.xlsx \
    --cofog "Defence" \
    --sector "General government" \
    --output-cols "DATA_DEFENCE"

### Individual Stages (Manual Execution)

In [ ]:
# Stage 1 & 2: Validation
!cofog-panel check-format --source-file ./data/gfs_raw_data.xlsx
!cofog-panel check-country-format --lookup-file ./data/country_codes.xlsx

In [ ]:
# Stage 3: Seeding the Master Schema
!cofog-panel seed-master --lookup-file ./data/country_codes.xlsx --output-dir ./output

In [20]:
# Stage 4: ETL and Data Splitting
!cofog-panel split --source-file ./data/gfs_raw_data.xlsx --cofog "{dropdown_cofog.value}" --filter-type "{dropdown_transformartion.value}"


--- ETL & SPLIT: Processing gfs_raw_data.xlsx ---
Filtering data by Transformation: Percent of GDP and COFOGs: {'Fuel and energy'}...

--- Filtering complete. Total Processed: 372482. Total Valid: 1552 ---
Writing 8 rows for country: AUT...
-> Successfully wrote AUT.
Writing 8 rows for country: SWE...
-> Successfully wrote SWE.
Writing 8 rows for country: GRC...
-> Successfully wrote GRC.
Writing 8 rows for country: LUX...
-> Successfully wrote LUX.
Writing 8 rows for country: BGR...
-> Successfully wrote BGR.
Writing 8 rows for country: CYP...
-> Successfully wrote CYP.
Writing 8 rows for country: CHE...
-> Successfully wrote CHE.
Writing 8 rows for country: GBR...
-> Successfully wrote GBR.
Writing 8 rows for country: UKR...
-> Successfully wrote UKR.
Writing 8 rows for country: AGO...
-> Successfully wrote AGO.
Writing 8 rows for country: SOM...
-> Successfully wrote SOM.
Writing 8 rows for country: BIH...
-> Successfully wrote BIH.
Writing 8 rows for country: PRY...
-> Successfull

In [21]:
# Stage 5: Harmonization and Aggregation
!cofog-panel aggregate --master-file ./output/COFOG_MASTER_SCHEMA.xlsx --folder-path ./intermediate_splits --data-col "{input_data_col.value}" --sector "{dropdown_sector.value}"


--- AGGREGATE STAGE: Processing files in intermediate_splits ---
✅ Master XLSX loaded: 8964 rows.
📂 Found 194 intermediate country XLSX files in intermediate_splits.
[1/194] Processing TUR -> OK (16 updates)
[2/194] Processing LVA -> OK (0 updates)
[3/194] Processing DMA -> OK (0 updates)
[4/194] Processing HND -> OK (0 updates)
[5/194] Processing OMN -> OK (0 updates)
[6/194] Processing SRB -> OK (6 updates)
[7/194] Processing PAN -> OK (0 updates)
[8/194] Processing BHS -> OK (0 updates)
[9/194] Processing BTN -> OK (0 updates)
[10/194] Processing CHN -> OK (10 updates)
[11/194] Processing EST -> OK (31 updates)
[12/194] Processing BRA -> OK (0 updates)
[13/194] Processing ABW -> OK (0 updates)
[14/194] Processing WBG -> OK (1 updates)
[15/194] Processing TZA -> OK (0 updates)
[16/194] Processing DNK -> OK (34 updates)
[17/194] Processing RUS -> OK (23 updates)
[18/194] Processing LUX -> OK (34 updates)
[19/194] Processing KNA -> OK (0 updates)
[20/194] Processing MRT -> OK (0 updat